#MTH 5000 – Applied Forecasting of Financial Data
##Lecture 3: Accessing and Retrieving Financial Data
##Part 2: yfinance api
---
Topics

*   1. Introduction to yfinance API
*   2. Loading data
*   3. YFinance examples
*   4. YFinance Template for Forecasting
*   5. Exporting Data   


#1. Introduction to yfinance API

In [6]:
#Loading and installing package
!pip install yfinance

In [11]:
#Check if package is loaded
!pip show yfinance

Name: yfinance
Version: 0.2.66
Summary: Download market data from Yahoo! Finance API
Home-page: https://github.com/ranaroussi/yfinance
Author: Ran Aroussi
Author-email: ran@aroussi.com
License: Apache
Location: /usr/local/lib/python3.12/dist-packages
Requires: beautifulsoup4, curl_cffi, frozendict, multitasking, numpy, pandas, peewee, platformdirs, protobuf, pytz, requests, websockets
Required-by: 


yfinance also has a few dependencies:

**pandas**:  
Converts downloaded financial data into tables (DataFrames) so we can analyze prices, returns, and volume.

**numpy**:  
Used for numerical operations and handling arrays efficiently.

**requests**:  
Sends HTTP requests to Yahoo Finance servers to fetch stock, ETF, and market data.

**lxml**:  
Parses HTML/XML when extracting structured data from Yahoo Finance pages.

**pytz**:  
Handles time zones correctly for datetime indices in historical data.

**datetime (Python stdlib)**:  
Used to manage start and end dates for downloading historical data.


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests
import lxml
import pytz
import datetime

##The following yfinance API methods will be used for data collection, feature engineering, and forecasting.


### Stock Price & Market Data (for Forecasting)
- `yf.download()` – Historical stock, ETF, or index prices (daily, weekly, monthly) used for time series forecasting.
- `Ticker.history()` – Historical OHLCV data for a single ticker (alternative to `yf.download()`).
- `Ticker.info['currentPrice']` – Real-time price (optional) for model evaluation or plotting.


#2. Loading data

## How to download historical data using the Yahoo Finance API

Historical price data is the one thing we will probably almost always need.

The method to get this in the `yfinance` library is `yf.download()` or `Ticker.history()`. We will have to import it from `yfinance`, so we do:



In [38]:
# Download daily Amazon data from 2010 to 2020
amazon_daily = yf.download(
    tickers="AMZN",
    start="2010-01-01",
    end="2020-01-01",
    interval="1d",
    auto_adjust=True
)

# Display first rows
amazon_daily.head()


[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,AMZN,AMZN,AMZN,AMZN,AMZN
Date,,,,,
2010-01-04,6.6950,6.8305,6.6570,6.8125,151998000
2010-01-05,6.7345,6.7740,6.5905,6.6715,177038000
2010-01-06,6.6125,6.7365,6.5825,6.7300,143576000
2010-01-07,6.5000,6.6160,6.4400,6.6005,220604000
2010-01-08,6.6760,6.6840,6.4515,6.5280,196610000


In [42]:
# Download daily Amazon data from 2010 to 2020
amazon_weekly = yf.download(
    tickers="AMZN",
    start="2010-01-01",
    end="2020-01-01",
    interval="1wk",
    auto_adjust=True
)

# Display first rows
amazon_weekly.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,AMZN,AMZN,AMZN,AMZN,AMZN
Date,,,,,
2010-01-01,6.5000,6.8305,6.4400,6.8125,693216000
2010-01-08,6.3675,6.6840,6.2875,6.5280,964086000
2010-01-15,6.3310,6.4825,6.2165,6.4590,866288000
2010-01-22,6.3015,6.3835,5.9060,6.2800,1504204000
2010-01-29,5.7970,6.5925,5.6910,6.4885,2310306000


Columns: Open, High, Low, Close, Volume

**We then pass in the following arguments to our function:**

- `tickers`: case-insensitive ticker symbol of the desired stock, ETF, or index (e.g., `"AMZN"`).

- `start`: start date of the data in `"YYYY-MM-DD"` format. Example: `"2009-12-04"`.

- `end`: end date of the data in `"YYYY-MM-DD"` format. Example: `"2019-12-04"`.

- `interval`: {`"1d"`, `"1wk"`, `"1mo"`}. Refers to the sampling interval of the data:
  - `"1d"` = daily
  - `"1wk"` = weekly
  - `"1mo"` = monthly

- `group_by`: {`"ticker"`, `"column"`}. Default is `"column"`. Determines how multiple tickers are grouped in the resulting DataFrame.

- `auto_adjust`: {True, False}. Default is True. If True, automatically adjusts OHLC for splits and dividends.

- `prepost`: {True, False}. Default is False. If True, includes pre-market and after-hours data for daily interval.

- `threads`: Number of threads to use for downloading multiple tickers. Default is True (auto).

#3. yfinance examples

Weekly data is often better for medium- to long-term forecasting:

In [36]:
# Download weekly data
apple_weekly = yf.download(
    tickers="AAPL",
    start="2025-01-01",
    end="2026-01-01",
    interval="1wk",
    auto_adjust=True
)

apple_weekly.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,AMZN,AMZN,AMZN,AMZN,AMZN
Date,,,,,
2010-01-01,6.5000,6.8305,6.4400,6.8125,693216000
2010-01-08,6.3675,6.6840,6.2875,6.5280,964086000
2010-01-15,6.3310,6.4825,6.2165,6.4590,866288000
2010-01-22,6.3015,6.3835,5.9060,6.2800,1504204000
2010-01-29,5.7970,6.5925,5.6910,6.4885,2310306000


You can download multiple tickers at once:

In [43]:
tickers = ["AMZN", "AAPL", "MSFT"]

data = yf.download(
    tickers=tickers,
    start="2010-01-01",
    end="2020-01-01",
    interval="1mo",  # Monthly data
    auto_adjust=True
)

data.head()


[*********************100%***********************]  3 of 3 completed


Price          Close                         High                     \
Ticker          AAPL    AMZN       MSFT      AAPL    AMZN       MSFT   
Date                                                                   
2010-01-01  5.760079  6.2705  21.059935  6.465769  6.8305  23.346783   
2010-02-01  6.136767  5.9200  21.426128  6.153262  6.2430  21.695170   
2010-03-01  7.047894  6.7885  21.991838  7.122273  6.9095  22.952901   
2010-04-01  7.830364  6.8550  22.930376  8.171361  7.5545  23.711239   
2010-05-01  7.704098  6.2730  19.371439  8.033999  6.9720  23.320810   

Price            Low                         Open                     \
Ticker          AAPL    AMZN       MSFT      AAPL    AMZN       MSFT   
Date                                                                   
2010-01-01  5.705796  5.9060  20.671319  6.400988  6.8125  22.883435   
2010-02-01  5.723790  5.6910  20.604058  5.769377  6.1590  21.216874   
2010-03-01  6.161659  5.8765  21.203465  6.170656  5.9350  21.601406   
2010-04-01  6.980417  6.5390  21.488781  7.120175  6.7900  22.036887   
2010-05-01  5.975714  5.8760  18.440409  7.912835  6.8600  23.027987   

Price            Volume                          
Ticker             AAPL        AMZN        MSFT  
Date                                             
2010-01-01  15168994400  4617220000  1359650900  
2010-02-01  10776080000  4202916000  1074643300  
2010-03-01  12154172800  3160852000  1110237200  
2010-04-01  12367129600  3460502000  1319029500  
2010-05-01  18082654800  2818198000  1720130200

Forecasting models usually work with a single time series column. For example, we can select just the Close prices:

In [44]:
# Daily closing prices
ts_daily = amazon_daily['Close']

# Weekly closing prices
ts_weekly = amazon_weekly['Close']

ts_weekly.head()

Ticker,AMZN
Date,
2010-01-01,6.5000
2010-01-08,6.3675
2010-01-15,6.3310
2010-01-22,6.3015
2010-01-29,5.7970


#4. YFinance Template for Forecasting

In [ ]:
# Inputs
TICKER = ""               # Stock/ETF/index ticker
START_DATE = "YEAR-MM-DD"     # Start date (YYYY-MM-DD)
END_DATE = "YEAR-MM-DD"       # End date (YYYY-MM-DD)
INTERVAL = ""              # Options: "1d", "1wk", "1mo"
AUTO_ADJUST = True            # Adjust OHLC for splits/dividends

# Download Historical Data
data = yf.download(
    tickers=TICKER,
    start=START_DATE,
    end=END_DATE,
    interval=INTERVAL,
    auto_adjust=AUTO_ADJUST
)

#5. Exporting Data   

Now that we have obtained our data it is important to know how to export it to other files in order to conduct forecasting.

## Option 1) Save the Data to your local machine as a CSV File

In [45]:
from google.colab import files

# Example: Download Amazon weekly data
amazon_weekly = yf.download("AMZN", start="2010-01-01", end="2020-01-01", interval="1wk")

# Save to CSV
amazon_weekly.to_csv("amazon_weekly.csv")

files.download("amazon_weekly.csv")

/tmp/ipython-input-373155958.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  amazon_weekly = yf.download("AMZN", start="2010-01-01", end="2020-01-01", interval="1wk")
[*********************100%***********************]  1 of 1 completed


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## To import the downloaded file

In [20]:
from google.colab import files

uploaded = files.upload()  # Select the CSV you just downloaded
import pandas as pd

amazon_weekly = pd.read_csv("amazon_weekly.csv", index_col=0, parse_dates=True)

/tmp/ipython-input-224941196.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  amazon_weekly = pd.read_csv("amazon_weekly.csv", index_col=0, parse_dates=True)


## Option 2) Save to Google Drive

In [21]:
from google.colab import drive
drive.mount('/content/drive')

# Save file to Drive
amazon_weekly.to_csv("/content/drive/MyDrive/amazon_weekly.csv")

Mounted at /content/drive


In [23]:
import pandas as pd

amazon_weekly = pd.read_csv("/content/drive/MyDrive/amazon_weekly.csv", index_col=0, parse_dates=True)

amazon_weekly

/tmp/ipython-input-2876190223.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  amazon_weekly = pd.read_csv("/content/drive/MyDrive/amazon_weekly.csv", index_col=0, parse_dates=True)


,Close,High,Low,Open,Volume
Price,,,,,
Ticker,AMZN,AMZN,AMZN,AMZN,AMZN
Date,NaN,NaN,NaN,NaN,NaN
2010-01-01,6.5,6.83050012588501,6.440000057220459,6.8125,693216000
2010-01-08,6.367499828338623,6.684000015258789,6.287499904632568,6.5279998779296875,964086000
2010-01-15,6.330999851226807,6.482500076293945,6.2164998054504395,6.459000110626221,866288000
...,...,...,...,...,...
2019-11-29,87.02400207519531,91.2344970703125,87.0,90.88899993896484,294476000
2019-12-06,88.0165023803711,88.34449768066406,86.75,87.55999755859375,265360000
2019-12-13,89.61399841308594,89.91000366210938,87.75,88.25,310790000


## Option 3) Using a .py file in Google Drive

In [33]:
from google.colab import files
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

Saving fetch_data.py to fetch_data.py
User uploaded file "fetch_data.py" with length 202 bytes


In [35]:
import fetch_data
amazon_weekly = fetch_data.get_amazon_weekly()
print(amazon_weekly.head())

[*********************100%***********************]  1 of 1 completed

Price        Close    High     Low    Open      Volume
Ticker        AMZN    AMZN    AMZN    AMZN        AMZN
Date                                                  
2010-01-01  6.5000  6.8305  6.4400  6.8125   693216000
2010-01-08  6.3675  6.6840  6.2875  6.5280   964086000
2010-01-15  6.3310  6.4825  6.2165  6.4590   866288000
2010-01-22  6.3015  6.3835  5.9060  6.2800  1504204000
2010-01-29  5.7970  6.5925  5.6910  6.4885  2310306000
